### Week 6, Day 2

We're about to create and use our own MCP Server and MCP Client!

It's pretty simple, but it's not super-simple. The excitment around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.

Let's review some python code made mostly by a hard-working Engineering Team:

accounts.py

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [2]:
# Here in the notebook are imported the function created in the file: accounts.py (our own python modules)
from accounts import Account

In [3]:
account = Account.get("Ed")
account

Account(name='ed', balance=9963.928, strategy='', holdings={'AMZN': 3}, transactions=[3 shares of AMZN at 12.024000000000001 each.], portfolio_value_time_series=[('2026-02-05 17:55:19', 10251.928), ('2026-02-05 17:55:23', 10155.928)])

In [4]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

'Completed. Latest details:\n{"name": "ed", "balance": 9771.544, "strategy": "", "holdings": {"AMZN": 6}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 12.024000000000001, "timestamp": "2026-02-05 17:55:19", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 64.128, "timestamp": "2026-02-10 09:09:47", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-02-05 17:55:19", 10251.928], ["2026-02-05 17:55:23", 10155.928], ["2026-02-10 09:09:47", 9795.544]], "total_portfolio_value": 9795.544, "total_profit_loss": -204.45600000000013}'

In [5]:
account.report()

'{"name": "ed", "balance": 9771.544, "strategy": "", "holdings": {"AMZN": 6}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 12.024000000000001, "timestamp": "2026-02-05 17:55:19", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 64.128, "timestamp": "2026-02-10 09:09:47", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-02-05 17:55:19", 10251.928], ["2026-02-05 17:55:23", 10155.928], ["2026-02-10 09:09:47", 9795.544], ["2026-02-10 09:09:49", 9807.544]], "total_portfolio_value": 9807.544, "total_profit_loss": -192.45600000000013}'

In [6]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 12.024000000000001,
  'timestamp': '2026-02-05 17:55:19',
  'rationale': 'Because this bookstore website looks promising'},
 {'symbol': 'AMZN',
  'quantity': 3,
  'price': 64.128,
  'timestamp': '2026-02-10 09:09:47',
  'rationale': 'Because this bookstore website looks promising'}]

### Now we write an MCP server and use it directly!

In [ ]:
# Now let's use our accounts server as an MCP server
# IMPORTANT: Use the full path to uv (same as we did for uvx)
# Also use the full path to accounts_server.py to ensure it's found

import os

accounts_server_path = os.path.abspath(os.path.join(os.getcwd(), "accounts_server.py"))
# The list below means "run the file accounts_server.py"
params = {"command": "/Users/federico.tognetti/.local/bin/uv", "args": ["run", accounts_server_path]}

# This creates an MCP Server
# We know we have some functions decorated as tools
# Once created the server we can list the tools
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

In [8]:
mcp_tools

[Tool(name='get_balance', description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, annotations=None),
 Tool(name='get_holdings', description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, annotations=None),
 Tool(name='buy_shares', description="Buy shares of a stock.\n\n    Args:\n        name: The name of the account holder\n        symbol: The symbol of the stock\n        quantity: The quantity of shares to buy\n        rationale: The rationale for the purchase and fit with the account's strategy\n    ", inputSchema={'properties': {'name': {'title': 'Name', '

In [9]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Ed and my account is under the name Ed. What's my balance and my holdings?"
model = "gpt-4.1-mini"

In [11]:
# The code below:
# 1. Launches an MCP server using the specified command parameters
# 2. Creates an Agent instance with the provided instructions and model, making it use the MCP server
# 3. Executes the request via the agent and displays the resulting output
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(starting_agent=agent, input=request)
    display(Markdown(result.final_output))


Ed, your account balance is $9,771.54. In terms of holdings, you currently have 6 shares of Amazon (AMZN). If you need any further assistance with your account, feel free to ask!

### Now let's build our own MCP Client

In [25]:
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

FileNotFoundError: [Errno 2] No such file or directory: 'uv'

In [ ]:
request = "My name is Ed and my account is under the name Ed. What's my balance?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

In [ ]:
context = await read_accounts_resource("ed")
print(context)

In [ ]:
from accounts import Account
Account.get("ed").report()

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Make your own MCP Server! Make a simple function to return the current Date, and expose it as a tool so that an Agent can tell you today's date.<br/>Harder optional exercise: then make an MCP Client, and use a native OpenAI call (without the Agents SDK) to use your tool via your client.
            </span>
        </td>
    </tr>
</table>

In [36]:
# Create a date server MCP server
# First, we need to set up the parameters to run date_server.py with uv
import os

date_server_path = os.path.abspath(os.path.join(os.getcwd(), "date_server.py"))
params = {"command": "/Users/federico.tognetti/.local/bin/uv", "args": ["run", date_server_path]}

instructions = "You are able to retrieve the current date and time"
request = "Please return the current date and time"
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as date_server:
    agent = Agent(name="date_teller", instructions=instructions, model=model, mcp_servers=[date_server])
    with trace("date_teller"):
        result = await Runner.run(starting_agent=agent, input=request)
    display(Markdown(result.final_output))


[02/10/26 10:57:31] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=727388;file:///Users/federico.tognetti/Desktop/AgenticAIEngineering/agents/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=104476;file:///Users/federico.tognetti/Desktop/AgenticAIEngineering/agents/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

[02/10/26 10:57:32] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=902403;file:///Users/federico.tognetti/Desktop/AgenticAIEngineering/agents/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=309375;file:///Users/federico.tognetti/Desktop/AgenticAIEngineering/agents/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

The current date and time is 2026-02-10T10:57:31.

[02/10/26 10:57:33] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=266084;file:///Users/federico.tognetti/Desktop/AgenticAIEngineering/agents/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=329352;file:///Users/federico.tognetti/Desktop/AgenticAIEngineering/agents/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       